# PUK-28 comparison runs on Colab

**What this notebook is.** The PUK-28 architecture comparison needed saved results that did not exist yet. This notebook
makes them. It was run once, on 2026-10-27, on a Colab **A100** GPU, and this copy keeps the output of that run.
The results are analysed in `docs/evaluation/ARCHITECTURE_COMPARISON.md`.

**What it makes (all on the same 30 gold topics x 4 mediums = 120 prompts):**

| Part | What it makes | Status in the saved run |
|---|---|---|
| A | Relabel adapter + RAG on, seeds **43** and **44** (120 drafts each) | Done, 120 of 120 drafts for both seeds |
| B | Scores for those drafts, judged by `qwen2.5:14b` | Done |
| C | "PersonaRAG on the new data" = base Llama 3.1 + train-only retrieval, seed 42 | Done, 120 of 120 drafts, scored |

Seed 42 of Relabel + RAG already existed (Kevin's run), so it was not repeated.

**Run the cells in order.** Everything is copied to Google Drive after each step, so a Colab disconnect does not lose finished work.

**Frozen on purpose (do not change):** the code at commit `89fe55e` (Kevin's final v1 pipeline; later commits switch the RAG
selector to v2), the 120 gold prompts, the 21,193-document `lemkin_train_only` index and the Relabel adapter from the handoff
zip, and the judge `qwen2.5:14b`. No data was rebuilt, so the new runs are comparable with Kevin's.

**What is different from Kevin's runs (say so when you quote the results).**
- GPU: A100 here, RTX 5070 Ti for Kevin. Python **3.13** here, 3.11 for Kevin. Torch, Unsloth and PEFT versions are the same (2.10.0+cu128, 2026.5.2, 0.19.1).
- Because of that, the extracted evidence differs from Kevin's seed 42 on 15 of 30 topics. Seeds 43 and 44 here have identical evidence, so they are a clean repeat of each other.
- Kevin's scoring script refuses any file except his seed-42 one. In this Colab copy only, step 9 switches that one check off. The change is recorded in `environment.json`.
- PersonaRAG here is an approximation (see the note in step 10).

## 1. Settings

In [1]:
ZIP_PATH = "/content/drive/MyDrive/298 B/relabel-aws-handoff-core-20261005.zip"
OUT_ROOT = "/content/drive/MyDrive/298 B/puk28_results"
REPO_URL = "https://github.com/andreahcruz/PersonaCloningGen.git"
COMMIT = "89fe55e"           # Kevin's final v1 pipeline (do not use the branch tip: it defaults to RAG v2)
SEEDS = [43, 44]             # Relabel + RAG seeds to generate (seed 42 already exists from Kevin)
RUN_PERSONARAG = True      # True = also run PersonaRAG on the new data (it was True in the saved run)
PERSONARAG_SEED = 42
EXPECTED_DOCS = 21193        # documents in lemkin_train_only

## 2. GPU and Google Drive

In [2]:
import os, subprocess, sys, json, time, hashlib, shutil, pathlib


def sh(cmd, cwd=None, env=None, check=True, log=None):
    """Run a shell command, stream its output, and optionally tee it to a log file."""
    full_env = dict(os.environ, **(env or {}))
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env=full_env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    fh = open(log, "a", encoding="utf-8") if log else None
    for line in proc.stdout:
        print(line, end="")
        if fh:
            fh.write(line)
    proc.wait()
    if fh:
        fh.close()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc.returncode


sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_ROOT, exist_ok=True)
assert os.path.isfile(ZIP_PATH), f"zip not found: {ZIP_PATH}"
print("zip size MB:", round(os.path.getsize(ZIP_PATH) / 1e6, 1))

name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
Mounted at /content/drive
zip size MB: 455.5


## 3. Unzip the handoff and check every file against its `SHA256SUMS`

In [5]:
import zipfile
root = pathlib.Path(HANDOFF_PARENT).resolve()
# The zip was made on Windows (backslash separators), so extract with Python and fix the separators.
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        name = info.filename.replace("\\", "/")
        target = (root / name).resolve()
        if not target.is_relative_to(root):
            raise RuntimeError(f"unsafe path in zip: {info.filename}")
        if name.endswith("/"):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)


In [6]:
sums = next(pathlib.Path(HANDOFF_PARENT).rglob("SHA256SUMS"))
HANDOFF = sums.parent
print("handoff root:", HANDOFF)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


bad, checked = [], 0
for line in sums.read_text(encoding="utf-8", errors="replace").splitlines():
    line = line.strip()
    if not line:
        continue
    digest, _, rel = line.partition(" ")
    rel = rel.strip().lstrip("*").replace("\\", "/")
    target = HANDOFF / rel
    if not target.is_file():
        bad.append(("missing", rel))
        continue
    checked += 1
    if sha256_file(target) != digest.lower():
        bad.append(("hash mismatch", rel))
print(f"checked {checked} files, problems: {len(bad)}")
for item in bad:
    print("  ", item)
assert not bad, "STOP: the handoff is incomplete or changed. Re-download it from Kevin."
ADAPTER = HANDOFF / "REQUIRED" / "lemkin_lora_relabel"
CHROMA = HANDOFF / "REQUIRED" / "chroma_lemkin_train_only"
assert (ADAPTER / "adapter_model.safetensors").is_file() and (CHROMA / "chroma.sqlite3").is_file()


handoff root: /content/handoff/relabel-aws-handoff-core-20261005
checked 36 files, problems: 0


## 4. Get the code at the frozen commit and install it

In [7]:
REPO = "/content/repo"
shutil.rmtree(REPO, ignore_errors=True)
LFS_OFF = {"GIT_LFS_SKIP_SMUDGE": "1"}     # skips ~480 MB of adapter files; we use the ones from the handoff zip
sh(f"git clone --filter=blob:none --no-checkout {REPO_URL} {REPO}", env=LFS_OFF)
sh(f"git checkout {COMMIT}", cwd=REPO, env=LFS_OFF)
sh("git log -1 --format='%H %ad %s' --date=short", cwd=REPO)
COMMIT_FULL = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert not os.path.exists(f"{REPO}/host_finetune/rag_active.json"), "this commit should be the v1 pipeline"

Cloning into '/content/repo'...
Note: switching to '89fe55e'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 89fe55e Document deployment setup and model artifacts
89fe55e5a2f82a5f3355a8c20a016d322a5a5c17 2026-10-05 Document deployment setup and model artifacts


In [8]:
# Install order matters (Kevin's requirements.txt explains why): torch from the cu128 index first.
sh('pip -q install "torch>=2.10.0,<2.11.0" "torchvision>=0.25.0,<0.26.0" --index-url https://download.pytorch.org/whl/cu128')
sh(f"pip -q install -r {REPO}/host_finetune/requirements.txt")
# Kevin's recorded versions. If this pin cannot be satisfied, drop it and write the versions in the report.
sh('pip -q install "unsloth==2026.5.2" "peft==0.19.1"', check=False)
sh('python -c "import torch, unsloth, peft, chromadb; print(torch.__version__, torch.cuda.is_available(), '
   'torch.cuda.get_device_name(0)); print(\'peft\', peft.__version__, \'chromadb\', chromadb.__version__)"')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuda-python 13.0.3 requires cuda-bindings~=13.0.3, but you have cuda-bindings 12.9.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Uns

0

## 5. Start Ollama (embeddings, and later the judge)

In [9]:
sh("apt-get -qq install -y zstd > /dev/null", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh")
import requests
ollama_log = open("/content/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama did not start; see /content/ollama.log")
sh("ollama pull nomic-embed-text")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                          

                                                                           0.0%
                                                                           0.7%
#                                                                          2.1%
##                                                                         4.1%
####                                                                       6.3%
#####                                                                      7.6%
#######                                                                    9.7%
########                                                                  11.8%
#########                                                                 13.3%
###########                                                               15.3%
############                            

0

In [10]:
sh("ollama --version")
sh("ollama show nomic-embed-text | head -20")

ollama version is 0.40.11
  Model
    architecture        nomic-bert    
    parameters          137M          
    context length      2048          
    embedding length    768           
    quantization        F16           

  Capabilities
    embedding    

  Parameters
    num_ctx    8192    

  License
    Apache License               
    Version 2.0, January 2004    
    ...                          



0

## 6. Check the index before spending GPU time

In [11]:
RUN_ENV = {
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
    "OLLAMA_BASE": "http://127.0.0.1:11434",
    "RELABEL_ADAPTER_PATH": str(ADAPTER),
    "CHROMA_PERSIST_DIR": str(CHROMA),
}
sh("python - <<'EOF'\n"
   "import chromadb, os\n"
   "c = chromadb.PersistentClient(path=os.environ['CHROMA_PERSIST_DIR']).get_collection('lemkin_train_only')\n"
   "print('lemkin_train_only documents:', c.count())\n"
   f"assert c.count() == {EXPECTED_DOCS}, 'unexpected document count'\n"
   "EOF", env=RUN_ENV)

lemkin_train_only documents: 21193


0

## 7. Part A: generate Relabel + RAG drafts for each seed

Same command as Kevin's final run, plus `--seed-base`. A seed is `seed_base + topic_index * 4 + medium_index`, so seed base 43
or 44 matches how his earlier seed runs were numbered. `--skip-score` writes the drafts only (no judge).
His first pass for seed 42 wrote 104 of 120 drafts and needed a targeted recovery; the next cell does the same if needed.

In [12]:
def run_dir(seed):
    return f"experiments/EXP-COLAB-relabel-rag-seed{seed}"


def count_rows(path):
    return sum(1 for line in open(path, encoding="utf-8") if line.strip()) if os.path.isfile(path) else 0


def save_to_drive(name):
    src = pathlib.Path(REPO) / "experiments" / name
    dst = pathlib.Path(OUT_ROOT) / name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("saved to Drive:", dst)


for seed in SEEDS:
    out = run_dir(seed)
    log = f"{REPO}/{out}.log"
    cmd = (f"python -m host_finetune.compare_adapters --adapters relabel={ADAPTER} --retrieval "
           f"--skip-score --seed-base {seed} --out-dir {out}")
    print("=== seed", seed, "\n", cmd)
    rc = sh(cmd, cwd=REPO, env=RUN_ENV, check=False, log=log)
    n = count_rows(f"{REPO}/{out}/raw/relabel.jsonl")
    print(f"seed {seed}: exit {rc}, {n}/120 drafts")
    save_to_drive(os.path.basename(out))
    if os.path.isfile(log):
        shutil.copy(log, OUT_ROOT)

=== seed 43 
 python -m host_finetune.compare_adapters --adapters relabel=/content/handoff/relabel-aws-handoff-core-20261005/REQUIRED/lemkin_lora_relabel --retrieval --skip-score --seed-base 43 --out-dir experiments/EXP-COLAB-relabel-rag-seed43
=== relabel /content/handoff/relabel-aws-handoff-core-20261005/REQUIRED/lemkin_lora_relabel
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloadi

## 8. Recovery for any missing drafts (skip if every seed shows 120/120)



In [ ]:
gold_ids = [json.loads(l)["id"] for l in open(f"{REPO}/data/openai_ft/lemkin_gold_eval.jsonl", encoding="utf-8") if l.strip()]
for seed in SEEDS:
    out = run_dir(seed)
    main_path = pathlib.Path(REPO) / out / "raw" / "relabel.jsonl"
    rows = [json.loads(l) for l in open(main_path, encoding="utf-8") if l.strip()] if main_path.exists() else []
    have = {(r["id"], r["medium"]) for r in rows}
    missing_topics = sorted({g for g in gold_ids if sum(1 for (i, _m) in have if i == g) < 4})
    print(f"seed {seed}: {len(rows)} rows, topics with missing drafts: {missing_topics}")
    if not missing_topics:
        continue
    rec = out + "-recovery"
    cmd = (f"python -m host_finetune.compare_adapters --adapters relabel={ADAPTER} --retrieval --skip-score "
           f"--seed-base {seed} --topic-ids {','.join(missing_topics)} --out-dir {rec}")
    sh(cmd, cwd=REPO, env=RUN_ENV, check=False, log=f"{REPO}/{rec}.log")
    rec_path = pathlib.Path(REPO) / rec / "raw" / "relabel.jsonl"
    new_rows = [json.loads(l) for l in open(rec_path, encoding="utf-8") if l.strip()] if rec_path.exists() else []
    merged = {(r["id"], r["medium"]): r for r in rows}
    merged.update({(r["id"], r["medium"]): r for r in new_rows})
    ordered = sorted(merged.values(), key=lambda r: r["seed"])
    shutil.copy(main_path, main_path.with_name("relabel.initial.jsonl"))
    with open(main_path, "w", encoding="utf-8") as f:
        f.write("\n".join(json.dumps(r, ensure_ascii=False) for r in ordered) + "\n")
    print(f"seed {seed}: merged to {len(ordered)}/120 drafts")
    save_to_drive(os.path.basename(out))
    save_to_drive(os.path.basename(rec))

## 9. Part B: score the drafts with the judge

**Read this first.** Kevin's scorer `score_saved_experiment.py` contains a check that refuses any file other than his
frozen seed 42 file (`EXPECTED_GENERATION_SHA`). To score the new seeds, this cell removes that one check in the
**Colab copy** of the code and changes nothing else. The patch is recorded in `environment.json`. If you or Kevin
prefer that he scores the drafts himself, set `SCORE = False` and send him the files instead.

In [13]:
import subprocess, time
print(subprocess.run("ollama pull qwen2.5:14b", shell=True, capture_output=True, text=True).stdout[-300:])
subprocess.run('ollama run qwen2.5:14b "say ok" --keepalive 5m', shell=True, capture_output=True, text=True)
print(subprocess.run("ollama ps", shell=True, capture_output=True, text=True).stdout)



NAME           ID              SIZE     PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:14b    7cdf5a0187d5    15 GB    100% GPU     32768      llamacpp    4 minutes from now    



In [14]:
SCORE = True
if SCORE:
    sh("ollama pull qwen2.5:14b")          # about 9 GB
    path = pathlib.Path(REPO) / "host_finetune" / "score_saved_experiment.py"
    text = path.read_text(encoding="utf-8")
    old = "if generation_sha != EXPECTED_GENERATION_SHA:"
    assert text.count(old) == 1, "scorer changed; do not patch blindly"
    path.write_text(text.replace(old, "if False:  # PATCHED in Colab: guard is hard-coded to the seed 42 file"), encoding="utf-8")
    for seed in SEEDS:
        out = run_dir(seed)
        traces = f"{out}/raw/relabel.jsonl"
        if count_rows(f"{REPO}/{traces}") != 120:
            print(f"seed {seed}: not 120 drafts, skipping scoring")
            continue
        sh(f"python -m host_finetune.score_saved_experiment --traces {traces} --out-dir {out} "
           f"--model-name relabel --source-experiment {os.path.basename(out)}",
           cwd=REPO, env=RUN_ENV, check=False, log=f"{REPO}/{out}.score.log")
        save_to_drive(os.path.basename(out))

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 2049f5674b1e: 100% ▕██████████████████▏ 9.0 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling db59b814cab7: 100% ▕██████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
judge=qwen2.5:14b family=qwen cached=0 generation_sha=32abc777fa8a96c6275a761e6895dc2e9c1eaf5e13200ef22fd7f6fe0f9c6ac5
held-out evidence docs=1603
writing calibrati

## 10. Part C: PersonaRAG on the new data

Base Llama 3.1 (through Ollama, no adapter) + dense retrieval from `lemkin_train_only`, same 120 prompts, seed 42.
This is the closest script match to "PersonaRAG on the new dataset", but it is **not** the old semester-1 `generate.py` pipeline,
so treat the PersonaRAG row as an approximation. The scorer was written for the Relabel evidence format; in the saved run it still
scored all 120 drafts (see the output below), but a team member who knows the old PersonaRAG setup should confirm that this is the right stand-in.

In [15]:
if RUN_PERSONARAG:
    sh("ollama pull llama3.1")
    out = "experiments/EXP-COLAB-personarag-newdata-seed42"
    cmd = (f"python -m host_finetune.compare_adapters --adapters none --ollama llama3.1 --retrieval "
           f"--skip-score --seed-base {PERSONARAG_SEED} --out-dir {out}")
    sh(cmd, cwd=REPO, env=RUN_ENV, check=False, log=f"{REPO}/{out}.log")
    print("drafts:", count_rows(f"{REPO}/{out}/raw/llama3.1.jsonl"), "/ 120")
    save_to_drive(os.path.basename(out))
    if SCORE:
        sh(f"python -m host_finetune.score_saved_experiment --traces {out}/raw/llama3.1.jsonl --out-dir {out} "
           f"--model-name llama3.1 --source-experiment {os.path.basename(out)}",
           cwd=REPO, env=RUN_ENV, check=False, log=f"{REPO}/{out}.score.log")
        save_to_drive(os.path.basename(out))
else:
    print("Part C skipped (RUN_PERSONARAG = False).")

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  44 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏ 113 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   5% ▕                  ▏ 244 MB/4.9 GB     

## 11. Record the environment and list what to send

In [17]:
env_info = {
    "date_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo": REPO_URL, "commit": COMMIT_FULL,
    "gpu": subprocess.check_output("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, text=True).strip(),
    "python": sys.version,
    "pip_freeze": subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines(),
    "handoff_zip": os.path.basename(ZIP_PATH),
    "judge_model": "qwen2.5:14b" if SCORE else "not run",
    "scoring_patch": "removed the hard-coded EXPECTED_GENERATION_SHA check in score_saved_experiment.py" if SCORE else "none",
    "seeds": SEEDS, "personarag_run": RUN_PERSONARAG,
}
json.dump(env_info, open(f"{OUT_ROOT}/environment.json", "w"), indent=2)
for p in sorted(pathlib.Path(OUT_ROOT).rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT_ROOT)}")
print("\nDownload the folder", OUT_ROOT, ".")

    0.00 MB  EXP-COLAB-personarag-newdata-seed42/config/evaluation_run.json
    0.00 MB  EXP-COLAB-personarag-newdata-seed42/config/run_parameters.json
    0.41 MB  EXP-COLAB-personarag-newdata-seed42/metrics/rows.jsonl
    0.00 MB  EXP-COLAB-personarag-newdata-seed42/metrics/scoring_skipped.json
    0.00 MB  EXP-COLAB-personarag-newdata-seed42/metrics/stance_calibration.json
    0.00 MB  EXP-COLAB-personarag-newdata-seed42/metrics/summary.json
    0.00 MB  EXP-COLAB-personarag-newdata-seed42/metrics/writing_calibration.json
    0.07 MB  EXP-COLAB-personarag-newdata-seed42/raw/evidence_review.jsonl
    0.39 MB  EXP-COLAB-personarag-newdata-seed42/raw/judge_cache.jsonl
    1.80 MB  EXP-COLAB-personarag-newdata-seed42/raw/llama3.1.jsonl
    0.00 MB  EXP-COLAB-relabel-rag-seed43/config/evaluation_run.json
    0.00 MB  EXP-COLAB-relabel-rag-seed43/config/run_parameters.json
    0.41 MB  EXP-COLAB-relabel-rag-seed43/metrics/rows.jsonl
    0.00 MB  EXP-COLAB-relabel-rag-seed43/metrics/scorin